# Sync Author Hidden Names (oxjob #1107)

Syncs "hide this name from the profile" curations from the users Heroku Postgres database to a Delta table. Runs in
`jobs/authors.yaml` upstream of `CreateAuthors`. Mirrors `SyncAuthorOrcidCurations`.

**Why:** a researcher who changed their name (a former name after a name change, ZD 5433 / 9026) asks that the old name
stop showing as an alternate name on their profile. The names are derived from the profile's works, so they came back
on every rebuild; a curation is the one place a correction can live (support-factory charter rule 6).

**Source** (typed view over `openalex_users.public.curations`, users-api migration for #1107):
- `author_alternate_name_removals`: entity `authors`, property `display_name_alternatives`, action `remove`;
  `hidden_name` = the name as served. Undo = delete the curation (the MERGE below deletes the row here).

**Target:** `openalex.authors.author_hidden_names`, one row per (author_id, name_key). `name_key` = the name's
letter/digit tokens, lowercased and sorted, so one curation also hides the same name in another order or with other
punctuation ('Smith, John Q.' = 'John Q. Smith' = 'John Q Smith').

**Effect** (`CreateAuthors`): only served fields change. `raw_author_names` (the API's `display_name_alternatives` and
`raw_author_names`) drops every name with a hidden key; a `full_name` that is a hidden name serves the profile's
`display_name` instead. Author matching (`authors_for_matching`) keeps every attested name, so works printed under the
old name still land on the same profile. The works' own bylines are the published record and are not touched.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.authors.author_hidden_names (
    author_id         BIGINT NOT NULL,
    name_key          STRING NOT NULL,
    hidden_name       STRING,
    curation_id       STRING,
    created           TIMESTAMP,
    updated_datetime  TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Preview: the latest curation per (author, name key)

In [ ]:
%sql
SELECT author_id, name_key, hidden_name, curation_id, created
  FROM (
    SELECT k.*, ROW_NUMBER() OVER (PARTITION BY author_id, name_key ORDER BY created DESC, curation_id DESC) AS rn
    FROM (
      SELECT author_id, TRIM(hidden_name) AS hidden_name, curation_id, created,
             ARRAY_JOIN(ARRAY_SORT(FILTER(SPLIT(LOWER(REGEXP_REPLACE(hidden_name, '[^\\p{L}\\p{N}]+', ' ')), ' '), t -> t <> '')), ' ') AS name_key
      FROM openalex_users.public.author_alternate_name_removals
      WHERE author_id IS NOT NULL AND hidden_name IS NOT NULL
    ) k
  )
  WHERE rn = 1 AND name_key <> ''

## Sync

`WHEN NOT MATCHED BY SOURCE THEN DELETE`: a deleted curation un-hides the name on the next CreateAuthors run.

In [ ]:
%sql
MERGE INTO openalex.authors.author_hidden_names AS target
USING (
  SELECT author_id, name_key, hidden_name, curation_id, created
    FROM (
      SELECT k.*, ROW_NUMBER() OVER (PARTITION BY author_id, name_key ORDER BY created DESC, curation_id DESC) AS rn
      FROM (
        SELECT author_id, TRIM(hidden_name) AS hidden_name, curation_id, created,
               ARRAY_JOIN(ARRAY_SORT(FILTER(SPLIT(LOWER(REGEXP_REPLACE(hidden_name, '[^\\p{L}\\p{N}]+', ' ')), ' '), t -> t <> '')), ' ') AS name_key
        FROM openalex_users.public.author_alternate_name_removals
        WHERE author_id IS NOT NULL AND hidden_name IS NOT NULL
      ) k
    )
    WHERE rn = 1 AND name_key <> ''
) AS source
ON target.author_id = source.author_id AND target.name_key = source.name_key
WHEN MATCHED AND NOT (target.hidden_name <=> source.hidden_name AND target.curation_id <=> source.curation_id) THEN UPDATE SET
  target.hidden_name      = source.hidden_name,
  target.curation_id      = source.curation_id,
  target.created          = source.created,
  target.updated_datetime = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (author_id, name_key, hidden_name, curation_id, created, updated_datetime)
  VALUES (source.author_id, source.name_key, source.hidden_name, source.curation_id, source.created, CURRENT_TIMESTAMP())
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify

In [ ]:
%sql
SELECT COUNT(*) AS hidden_names, COUNT(DISTINCT author_id) AS profiles, MAX(updated_datetime) AS last_sync
FROM openalex.authors.author_hidden_names